# Interactive map showing AOIs associated with each CSDA Evaluation Site  

*Notes:*  
+ AOIs for CSDA sites (a lat, lon):
  + default to 3 km radius circles
  + can adjusted to be 3 km length boxes
  + can be a custom AOI (provide me with a single custom AOI as a geojson file).

Paul Montesano, PhD  
2026

In [39]:
library(sf)
library(leaflet)
library(dplyr)

In [40]:
# Read the GeoJSON file
sites <- st_read("../sites/eval_sites_aoi.geojson")

Reading layer `eval_sites_aoi' from data source 
  `/panfs/ccds02/home/pmontesa/code/csda_summaries/sites/eval_sites_aoi.geojson' 
  using driver `GeoJSON'
Simple feature collection with 555 features and 18 fields
Geometry type: POLYGON
Dimension:     XY
Bounding box:  xmin: -157.2105 ymin: -71.97737 xmax: 150.0512 ymax: 67.42133
Geodetic CRS:  WGS 84


In [41]:
colnames(sites)

[1] "Site.Name.abbrev"      "Site.Name"             "Location.Name"        
 [4] "Country"               "Program.Use"           "Longitude"            
 [7] "Latitude"              "Remote.Sensing.Domain" "Priority.Level"       
[10] "Evaluation.Category"   "Source"                "Surface.Domain"       
[13] "Assessment.type.s."    "aoi_type"              "aoi_size_km"          
[16] "Contact"               "Reference"             "area_km2"             
[19] "geometry"

In [63]:
# Create 25 km radius circles to show the vicinity around each CSDA site centroid
sites_buffered <- sites %>%
  st_transform(4326) %>%  # Ensure WGS84
  st_centroid() %>%
  st_transform(3857) %>%  # Transform to metric CRS for buffering
  st_buffer(dist = 25000) %>%  # 25 km = 25,000 meters
  st_transform(4326)  # Transform back to WGS84 for leaflet

# Create color palette
eval_categories <- unique(sites$Remote.Sensing.Domain)
color_pal <- colorFactor("Set3", domain = eval_categories)

# Create an interactive map
csda_sites_map = leaflet(height=200) %>%
  addProviderTiles(providers$Esri.WorldGrayCanvas, group = "Basemap") %>%
  addProviderTiles(providers$OpenStreetMap, group = "Street Map") %>%
  addProviderTiles(providers$Esri.WorldImagery, group = "Satellite") %>%
  addProviderTiles(providers$Esri.WorldTerrain, group="Terrain") %>%

  # Add 25 km radius circles (drawn first so they're behind)
  addPolygons(
    data = sites_buffered,
    fillColor = "transparent",
    fillOpacity = 0,
    color = "black",
    weight = 1,
    dashArray = "5, 5",  # Dotted line
    opacity = 0.7,
    group = "25 km Buffers"
  ) %>%
  
  # Add actual site polygons
  addPolygons(
    data = sites,
    fillColor = ~color_pal(Remote.Sensing.Domain),
    fillOpacity = 0.7,
    color = "white",
    weight = 2,
    popup = ~paste0(
      "<b>Site Name:</b> ", Site.Name, "<br>",
      "<b>SME Domain:</b> ", Remote.Sensing.Domain, "<br>",
      "<b>Area (km²):</b> ", round(st_area(geometry) / 1e6, 2)
    ),
    label = ~Site.Name,
    highlightOptions = highlightOptions(
      weight = 3,
      color = "red",
      fillOpacity = 0.9,
      bringToFront = TRUE
    ),
    group = "AOIs of CSDA Sites"
  ) %>%
  
  # Add layers control
  addLayersControl(
    baseGroups = c("Basemap","Satellite", "Street Map","Terrain"),
    overlayGroups = c("AOIs of CSDA sites", "Vicinity of CSDA Sites" ),
    options = layersControlOptions(collapsed = FALSE)
  ) %>%
  
  # Add legend
  addLegend(
    position = "bottomright",
    pal = color_pal,
    values = sites$Remote.Sensing.Domain,
    title = "SME Domain",
    opacity = 0.7
  ) %>%

  addScaleBar(
      position = "bottomleft",  # or "topleft", "topright", "bottomright"
      options = scaleBarOptions(
        maxWidth = 100,         # width in pixels
        metric = TRUE,          # show metric units (km/m)
        imperial = TRUE,        # show imperial units (mi/ft)
        updateWhenIdle = FALSE  # update during zoom/pan
      )
    )

Warning message:
“st_centroid assumes attributes are constant over geometries”


In [67]:
library(htmlwidgets)

# First save just the map
saveWidget(csda_sites_map, file = "csda_eval_sites_map.html", selfcontained = TRUE)

# Read the HTML
map_html <- readLines("csda_eval_sites_map.html")

# Find insertion point (try multiple patterns)
body_line <- grep("<body", map_html, ignore.case = TRUE)[1]
if (is.na(body_line)) {
  body_line <- grep("<div", map_html)[1]  # Fallback
}

# Create header HTML
header <- c(
  "<div style='padding: 20px; font-family: Arial, sans-serif;'>",
  "<h1>Interactive map showing AOIs associated with each CSDA Evaluation Site</h1>",
  "<p>This interactive map shows the CSDA evaluation sites with 25 km buffer zones. Sites are colored by evaluation category. Click on sites for more details.</p>",
  "<p>This is a work in progress - AOIs can be added/updated.</p>",
  "<p>Contact: paul.m.montesano@nasa.gov</p>",
    "<hr>",
  "</div>"
)

# Insert header
if (!is.na(body_line)) {
  updated_html <- c(
    map_html[1:body_line],
    header,
    map_html[(body_line+1):length(map_html)]
  )
  writeLines(updated_html, "csda_eval_sites_map.html")
} else {
  print("Could not find insertion point")
  print(head(map_html, 20))  # Show first 20 lines for debugging
}